In [5]:
import re
from pathlib import Path

# Секції (заголовки + дані), які ВИДАЛЯЄМО
SECTIONS_TO_DELETE_BLOCKS = (
    "Masses",
    "Pair Coeffs",
    "Bond Coeffs",
    "Angle Coeffs",
    "Dihedral Coeffs",
    "Improper Coeffs",
    "Velocities",
)

# Рядки заголовка, які ВИДАЛЯЄМО (тому що LAMMPS їх додасть)
HEADER_LINES_TO_DELETE_KEYWORDS = [
    "atoms",
    "bonds",
    "angles",
    "dihedrals",
    "impropers",
    "atom types",
    "bond types",
    "angle types",
    "dihedral types",
    "improper types",
]

# Секції (лише заголовки), які ЗАЛИШАЄМО
SECTIONS_TO_KEEP_HEADERS = (
    "Atoms",
    "Bonds",
    "Angles",
    "Dihedrals",
    "Impropers",
)

def _is_block_to_delete(line: str) -> bool:
    s = line.strip().casefold()
    return any(s.startswith(t.casefold()) for t in SECTIONS_TO_DELETE_BLOCKS)

def _is_header_to_keep(line: str) -> bool:
    s = line.strip().casefold()
    return any(s.startswith(t.casefold()) for t in SECTIONS_TO_KEEP_HEADERS)

def _is_header_line_to_delete(line: str) -> bool:
    s = line.strip().casefold()
    # Check if line contains any keyword, e.g., "74670 atoms"
    for keyword in HEADER_LINES_TO_DELETE_KEYWORDS:
        if keyword in s:
            # Make sure it's not a block header we want to keep
            if not _is_header_to_keep(line):
                return True
    return False

def clean_files_for_merge(in_path: str | Path, out_path: str | Path):
    in_path = Path(in_path)
    out_path = Path(out_path)
    lines = in_path.read_text(encoding="utf-8", errors="ignore").splitlines(keepends=False)

    output_lines = []
    is_in_drop_section = False

    for line in lines:
        
        # 1. Check if it's a block to delete (e.g., "Masses")
        if _is_block_to_delete(line):
            is_in_drop_section = True
            continue # Skip line
            
        # 2. Check if it's a data block header (e.g., "Atoms")
        if _is_header_to_keep(line):
            is_in_drop_section = False # Stop skipping
            output_lines.append(line)
            continue
            
        # 3. Check if it's a header count line (e.g., "74670 atoms")
        if _is_header_line_to_delete(line):
            continue # Skip line

        # 4. Handle data lines
        if is_in_drop_section:
            if line.strip() == "": # End of block
                is_in_drop_section = False
            continue # Skip data
            
        # 5. Keep the rest (e.g., box dims, atom data)
        output_lines.append(line)

    # Re-join and write the file
    out_text = "\n".join(output_lines).rstrip() + "\n"
    out_text = re.sub(r"\n\n+", "\n\n", out_text) # Clean up extra empty lines
    out_path.write_text(out_text, encoding="utf-8")

# --- Main Execution ---
print("Running FINAL cleaning script...")

# Використовуйте ОРИГІНАЛЬНІ файли (.lammps) як вхідні
WATER_IN = "/mnt/c/Users/vbarv/Desktop/vns/course/project/lammps/data/water_cylinder.data"
WATER_OUT = "/mnt/c/Users/vbarv/Desktop/vns/course/project/lammps/data/water_cylinder_FINAL_CLEAN.data"

GRAPH_IN = "/mnt/c/Users/vbarv/Desktop/vns/course/project/lammps/data/graphene_flat_initial.data"
GRAPH_OUT = "/mnt/c/Users/vbarv/Desktop/vns/course/project/lammps/data/graphene_flat_initial_FINAL_CLEAN.data"

clean_files_for_merge(WATER_IN, WATER_OUT)
clean_files_for_merge(GRAPH_IN, GRAPH_OUT)

print(f"✅ FINAL Cleaned: {Path(WATER_OUT).name} and {Path(GRAPH_OUT).name}")

Running FINAL cleaning script...
✅ FINAL Cleaned: water_cylinder_FINAL_CLEAN.data and graphene_flat_initial_FINAL_CLEAN.data
